<a href="https://colab.research.google.com/github/rajiv-ranjan/cds-mini-projects/blob/Archana/M6_NB_MiniProject_1_Medical_Q%26A_GPT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Certification Program in Computational Data Science
## A programme by IISc and TalentSprint
### Mini-Project: Medical Q&A using GPT2

## Learning Objectives

At the end of the experiment, you will be able to:

* perform data preprocessing, EDA and feature extraction on the Medical Q&A dataset
* load a pre-trained tokenizer
* finetune a GPT-2 language model for medical question-answering

## Dataset Description

The dataset used in this project is the *Medical Question Answering Dataset* ([MedQuAD](https://github.com/abachaa/MedQuAD/tree/master)). It includes medical question-answer pairs along with additional information, such as the question type, the question *focus*, its UMLS(Unified Medical Language System) details like - Concept Unique Identifier(*CUI*) and Semantic *Type* and *Group*.

To know more about this data's collection, and construction method, refer to this [paper](https://bmcbioinformatics.biomedcentral.com/articles/10.1186/s12859-019-3119-4).

The data is extracted and is in CSV format with below features:

- **Focus**: the question focus
- **CUI**: concept unique identifier
- **SemanticType**
- **SemanticGroup**
- **Question**
- **Answer**

## Part-A: Grading = 10 Points

## Information

Healthcare professionals often have to refer to medical literature and documents while seeking answers to medical queries. Medical databases or search engines are powerful resources of upto date medical knowledge. However, the existing documentation is large and makes it difficult for professionals to retrieve answers quickly in a clinical setting. The problem with search engines and informative retrieval engines is that these systems return a list of documents rather than answers. Instead, healthcare professionals can use question answering systems to retrieve short sentences or paragraphs in response to medical queries. Such systems have the biggest advantage of generating answers and providing hints in a few seconds.

### Problem Statement

Fine-tune gpt2 model on medical-question-answering-dataset for performing response generation for medical queries.

Please refer to ***M6 Assignment-1 Fine-tune GPT2*** to get familiar with how to load pre-trained gpt2 tokenizer and model.

### Import required packages

In [1]:
!pip -q install -U accelerate
!pip -q install -U transformers
!pip -q install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 127.3 MB/s eta 0:00:00


In [2]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TextDataset, DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments

import warnings
warnings.filterwarnings('ignore')

In [3]:
#@title Download the dataset
!wget -q https://cdn.iisc.talentsprint.com/AIandMLOps/MiniProjects/Datasets/MedQuAD.csv
!ls | grep ".csv"

MedQuAD.csv


**Exercise 1: Read the MedQuAD.csv dataset**

**Hint:** pd.read_csv()

In [4]:
df = pd.read_csv("MedQuAD.csv")
df.shape

(16412, 6)

In [5]:
df.head()

,Focus,CUI,SemanticType,SemanticGroup,Question,Answer
0,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,What is (are) Adult Acute Lymphoblastic Leukem...,Key Points - Adult acute lymphoblastic leukemi...
1,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,What are the symptoms of Adult Acute Lymphobla...,"Signs and symptoms of adult ALL include fever,..."
2,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,How to diagnose Adult Acute Lymphoblastic Leuk...,Tests that examine the blood and bone marrow a...
3,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,What is the outlook for Adult Acute Lymphoblas...,Certain factors affect prognosis (chance of re...
4,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,Who is at risk for Adult Acute Lymphoblastic L...,Previous chemotherapy and exposure to radiatio...


### Pre-processing and EDA

**Exercise 2: Perform below operations on the dataset [0.5 Mark]**

- Handle missing values
- Remove duplicates from data considering `Question` and `Answer` columns

- **Handle missing values**

In [6]:
# Check missing values
df.isnull().sum()

,0
Focus,14
CUI,565
SemanticType,597
SemanticGroup,565
Question,0
Answer,5


In [7]:
# Drop missing values
df.dropna(subset=["Answer"], inplace=True)

In [8]:
df.isnull().sum()

,0
Focus,14
CUI,565
SemanticType,597
SemanticGroup,565
Question,0
Answer,0


- **Remove duplicates from data considering `Question` and `Answer` columns**

In [9]:
# Check duplicates
df.duplicated(subset=['Question', 'Answer']).sum()

np.int64(48)

In [10]:
# Drop duplicates
df.drop_duplicates(subset=['Question', 'Answer'], inplace=True)

In [11]:
# Check duplicates
df.duplicated(subset=['Question', 'Answer']).sum()

np.int64(0)

**Exercise 3: Display the category name, and the number of records belonging to top 100 categories of `Focus` column [1 Mark]**

In [12]:
# YOUR CODE HERE
categoryName = df['Focus'].value_counts().index
categoryName

Index(['Breast Cancer', 'Prostate Cancer', 'Stroke', 'Skin Cancer',
       'Alzheimer's Disease', 'Lung Cancer', 'Colorectal Cancer',
       'High Blood Cholesterol', 'Heart Attack', 'Heart Failure',
       ...
       'Ichthyosis hystrix gravior', 'Ichthyosis hystrix, Curth Macklin type',
       'Ichthyosis prematurity syndrome', 'Ichthyosis, acquired',
       'Hypothalamic hamartomas', 'I cell disease', 'ICF syndrome',
       'Ichthyosiform erythroderma, corneal involvement, deafness',
       'Ichthyosis alopecia eclabion ectropion mental retardation',
       'Ichthyosis follicularis atrichia photophobia syndrome'],
      dtype='object', name='Focus', length=5125)

In [13]:
# Top 100 Focus categories names
df['Focus'].value_counts().head(100)

,count
Focus,
Breast Cancer,53
Prostate Cancer,43
Stroke,35
Skin Cancer,34
Alzheimer's Disease,30
...,...
MECP2 duplication syndrome,11
Holt-Oram syndrome,11
Ehlers-Danlos syndrome,11


### Create Training and Validation set

**Exercise 4: Create training and validation set [2 Marks]**

- Consider 4 samples per `Focus` category, for each top 100 categories, from the dataset (It will give 400 samples for training)

- Consider 1 sample per `Focus` category (different from training set), for each top 100 categories, from the dataset (It will give 100 samples for validation)

In [14]:
# create list of top 100 categories
top_Focus = df['Focus'].value_counts().head(100).index.to_list()
top_Focus

['Breast Cancer',
 'Prostate Cancer',
 'Stroke',
 'Skin Cancer',
 "Alzheimer's Disease",
 'Lung Cancer',
 'Colorectal Cancer',
 'High Blood Cholesterol',
 'Heart Attack',
 'Heart Failure',
 'High Blood Pressure',
 "Parkinson's Disease",
 'Leukemia',
 'Osteoporosis',
 'Shingles',
 'Age-related Macular Degeneration',
 'Diabetes',
 'Hemochromatosis',
 'Diabetic Retinopathy',
 'Psoriasis',
 'Gum (Periodontal) Disease',
 'Kidney Disease',
 'COPD',
 'Cataract',
 'Balance Problems',
 'Dry Mouth',
 'Prescription and Illicit Drug Abuse',
 'Medicare and Continuing Care',
 'Gout',
 'Glaucoma',
 'Wilson Disease',
 'Problems with Taste',
 'Neuroblastoma',
 'Rheumatoid Arthritis',
 'Short Bowel Syndrome',
 'Osteoarthritis',
 'Narcolepsy',
 'Endometrial Cancer',
 'Pituitary Tumors',
 'Dry Eye',
 'Kidney Dysplasia',
 'Anxiety Disorders',
 'Peripheral Arterial Disease (P.A.D.)',
 'Urinary Tract Infections in Children',
 'Surviving Cancer',
 'Problems with Smell',
 'Knee Replacement',
 'Creating a Famil

In [44]:
# YOUR CODE HERE
train_samples = []
val_samples = []

for focus in top_Focus:
    focus_df = df[df['Focus'] == focus]

    focus_df = focus_df.sample(n=5, random_state=42)
    train_samples.append(focus_df.iloc[:4])
    val_samples.append(focus_df.iloc[4:5])

train_df = pd.concat(train_samples).reset_index(drop=True)
val_df = pd.concat(val_samples).reset_index(drop=True)

print(len(train_df))
print(len(val_df))


400
100


### Pre-process `Question` and `Answer` text

**Exercise 5: Perform below tasks: [1.5 Marks]**

- Combine `Question` and `Answer` for train and validation data as shown below:
    - sequence = *'\<question\>' + question-text + '\<answer\>' + answer-text*

- Join the combined text using '\n' into a single string for training and validation separately

- Save the training and validation strings as separate text files

- **Combine Question and Answer for train and val data**

In [45]:
train_df['text'] = '<question>' + train_df['Question'] + '<answer>' + train_df['Answer']
train_df.head()

,Focus,CUI,SemanticType,SemanticGroup,Question,Answer,text
0,Breast Cancer,C0006142,T191,Disorders,Who is at risk for Breast Cancer? ?,Key Points - Avoiding risk factors and increas...,<question>Who is at risk for Breast Cancer? ?<...
1,Breast Cancer,C0006142,T191,Disorders,What is (are) Breast Cancer ?,A mammogram can often detect breast changes in...,<question>What is (are) Breast Cancer ?<answer...
2,Breast Cancer,C0006142,T191,Disorders,What is (are) Breast Cancer ?,There are two types of breast-conserving surge...,<question>What is (are) Breast Cancer ?<answer...
3,Breast Cancer,C0006142,T191,Disorders,What are the symptoms of Breast Cancer ?,Signs of breast cancer include a lump or chang...,<question>What are the symptoms of Breast Canc...
4,Prostate Cancer,C0376358,T191,Disorders,What is (are) Prostate Cancer ?,Surgery is a common treatment for early stage ...,<question>What is (are) Prostate Cancer ?<answ...


In [46]:
print(train_df['text'][0])

<question>Who is at risk for Breast Cancer? ?<answer>Key Points - Avoiding risk factors and increasing protective factors may help prevent cancer. - The following are risk factors for breast cancer: - Older age - A personal history of breast cancer or benign (noncancer) breast disease - Inherited risk of breast cancer - Dense breasts - Exposure of breast tissue to estrogen made in the body - Taking hormone therapy for symptoms of menopause - Radiation therapy to the breast or chest - Obesity - Drinking alcohol - The following are protective factors for breast cancer: - Less exposure of breast tissue to estrogen made by the body - Taking estrogen-only hormone therapy after hysterectomy, selective estrogen receptor modulators, or aromatase inhibitors and inactivators - Estrogen-only hormone therapy after hysterectomy - Selective estrogen receptor modulators - Aromatase inhibitors and inactivators - Risk-reducing mastectomy - Ovarian ablation - Getting enough exercise - It is not clear wh

In [48]:
val_df['text'] = '<question>' + val_df['Question'] + '<answer>' + val_df['Answer']
val_df.head()

,Focus,CUI,SemanticType,SemanticGroup,Question,Answer,text
0,Breast Cancer,C0006142,T191,Disorders,What are the treatments for Breast Cancer ?,You can seek conventional treatment from a spe...,<question>What are the treatments for Breast C...
1,Prostate Cancer,C0376358,T191,Disorders,What are the treatments for Prostate Cancer ?,There are a number of ways to treat prostate c...,<question>What are the treatments for Prostate...
2,Stroke,C0038454,T047,Disorders,Who is at risk for Stroke? ?,A risk factor is a condition or behavior that ...,<question>Who is at risk for Stroke? ?<answer>...
3,Skin Cancer,C0007114,T191,Disorders,How to prevent Skin Cancer ?,Key Points - Avoiding risk factors and increas...,<question>How to prevent Skin Cancer ?<answer>...
4,Alzheimer's Disease,C0002395,T046,Disorders,How to prevent Alzheimer's Disease ?,"Currently, no medicines or other treatments ar...",<question>How to prevent Alzheimer's Disease ?...


- **Join the combined text using '\n' into a single string for training and validation separately**

In [49]:
train_combined_text = '\n'.join(train_df['text'].to_list())
val_combined_text = '\n'.join(val_df['text'].to_list())

print(len(train_combined_text))
print(len(val_combined_text))

569069
147898


- **Save the training and validation strings as text files**

In [51]:
with open('train_data.txt', 'w') as f:
    f.write(train_combined_text)

with open('val_data.txt', 'w') as f:
    f.write(val_combined_text)

**Exercise 6: Load pre-trained GPT2Tokenizer [0.5 Mark]**

- Use checkpoint = "gpt2"

In [52]:
checkpoint = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(checkpoint)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

**Exercise 7: Tokenize train and validation data and form TextDataset objects [0.5 Mark]**

- Use the loaded pre-trained tokenizer
- Use training and validation data saved in text files

In [54]:
# Tokenize train text
train_dataset = TextDataset(tokenizer=tokenizer, file_path="train_data.txt", block_size=128)

# Tokenize validation text
val_dataset = TextDataset(tokenizer=tokenizer, file_path="val_data.txt", block_size=128)

In [55]:
print(len(train_dataset))
print(len(val_dataset))

936
246


In [56]:
# Batch-size
train_dataset[0].shape, val_dataset[0].shape

(torch.Size([128]), torch.Size([128]))

**Exercise 8: Create a DataCollator object [0.5 Mark]**

In [57]:
# Create a Data collator object
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, return_tensors="pt")

**Exercise 9: Load pre-trained GPT2LMHeadModel [0.5 Mark]**

In [58]:
# Set up the model
model = GPT2LMHeadModel.from_pretrained(checkpoint)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

**Exercise 10: Fine-tune GPT2 Model [1 Mark]**

- Specify training arguments and create a TrainingArguments object (Use 30 epochs)

- Train a GPT-2 model using the provided training arguments

- Save the resulting trained model and tokenizer to a specified output directory

In [59]:
# Set up the training arguments

model_output_path = "/content/gpt_model"

training_args = TrainingArguments(
    output_dir = model_output_path,
    overwrite_output_dir = True,
    per_device_train_batch_size = 4, # try with 2
    per_device_eval_batch_size = 4,  #  try with 2
    num_train_epochs = 30,
    save_steps = 1_000,
    save_total_limit = 2,
    logging_dir = './logs',
    )

In [60]:
# Train the model
trainer = Trainer(
    model = model,
    args = training_args,
    data_collator = data_collator,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
)

trainer.train()

# Save the model
trainer.save_model(model_output_path)

# Save the tokenizer
tokenizer.save_pretrained(model_output_path)

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: archanaraghav86 (archanaraghav86-indian-institute-of-science) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,2.491500
1000,1.937300
1500,1.579500
2000,1.279400
2500,1.037900
3000,0.855600
3500,0.707800
4000,0.596600
4500,0.509300
5000,0.446800


('/content/gpt_model/tokenizer_config.json',
 '/content/gpt_model/special_tokens_map.json',
 '/content/gpt_model/vocab.json',
 '/content/gpt_model/merges.txt',
 '/content/gpt_model/added_tokens.json')

**Exercise 11: Test Model with user input prompts [1 Mark]**

- Create `generate_response()` function that takes a trained *model*, *tokenizer*, and a *prompt* string as input and generates a response using the GPT-2 model

- Test it with some user input prompts

In [61]:
# YOUR CODE HERE
def generate_response(model, tokenizer, prompt, max_length=100):

    input_ids = tokenizer.encode(prompt, return_tensors="pt")      # 'pt' for returning pytorch tensor

    # Create the attention mask and pad token id
    attention_mask = torch.ones_like(input_ids)
    pad_token_id = tokenizer.eos_token_id

    output = model.generate(
        input_ids,
        max_length=max_length,
        num_return_sequences=1,
        attention_mask=attention_mask,
        pad_token_id=pad_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [62]:
# Load the fine-tuned model and tokenizer

my_model = GPT2LMHeadModel.from_pretrained(model_output_path)
my_tokenizer = GPT2Tokenizer.from_pretrained(model_output_path)

In [63]:
# Response from model

prompt = "Who is at risk for Stroke?"
response = generate_response(my_model, my_tokenizer, prompt)
print("Generated response:", response)

Generated response: Who is at risk for Stroke? Approximately 15 percent, or 36 million, of American adults age 20 years or older will have a stroke at some point in their lives. That's one stroke every 2 minutes. About one-third of all older adults in the United States will have a stroke at some point in their lives. If you have a stroke, it is very important to get treatment right away. If you have symptoms of stroke and are not sure how to care for yourself, get early


In [64]:
# Testing with given prompt 1

prompt = "How to prevent Skin Cancer ?"
response = generate_response(my_model, my_tokenizer, prompt)
print("Generated response:", response)

Generated response: How to prevent Skin Cancer ?<answer>Causes of Skin Cancer can range from simple genetic causes such as environmental factors, diseases of the skin, or environmental toxins to serious health problems such as cancer. Recognizing these causes can help you prevent skin cancer. Treatments include creams and medicines. Your doctor can prescribe these if you have a problem with your skin or if you smoke. NIH: National Cancer Institute
<question>What are the treatments for Skin Cancer ?<answer>Certain medicines


In [65]:
# Testing with given prompt 2

prompt = "What are the treatment for cancer?"
response = generate_response(my_model, my_tokenizer, prompt)
print("Generated response:", response)

Generated response: What are the treatment for cancer? There are different kinds of chemotherapy and medicines that can stop or slow down the progression of cancer. The goals of treating cancer are to: Stop the spread of the disease (cancer) by stopping the production of too many cells (cancer) by stopping the production of too many abnormal cells (cancer) by stopping the production of enough abnormal proteins (cancer) by stopping the production of too many abnormal cells (cancer) by stopping the production of too many abnormal cells (cancer


**Exercise 12: Compare the performance of a *GPT2 model* with the *GPT2 model fine-tuned* on MedQuAD data [1 Mark]**

- Load another pre-trained GPT2LMHeadModel and do not fine-tune it

- To generate response using the untuned model, pass it as a parameter to `generate_response()` function

- Test both models (fine-tuned and untuned) with below user input prompts:

    - "What precautions to take for a healthy life?"
    - "What to do after being diagnosed with cancer?"
    - "What to do when feeling sick?"

In [66]:
# Load a pre-trained GPT2 model, do not finetune it with MedQuAD data

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

In [67]:
# Testing with finetuned model: prompt 1

prompt = "What are the symtoms of tuberculosis?"
response = generate_response(my_model, my_tokenizer, prompt)
print("Generated response:", response)

Generated response: What are the symtoms of tuberculosis? ?<answer>Babies with TB often have no signs or symptoms. If TB is serious, it can affect the entire body. The signs and symptoms of TB include fevers, frequent infections, fatigue, muscle pain, diarrhea, and vomiting. People with TB may also have rash or fever. These symptoms may be similar to those of other conditions. They include - fevers - frequent infections - fatigue - muscle pain - vomiting - The first symptoms of


In [68]:
# Testing with untuned model: prompt 1

prompt = "What are the symtoms of tuberculosis?"
response = generate_response(model, tokenizer, prompt)
print("Generated response:", response)

Generated response: What are the symtoms of tuberculosis?

Tuberculosis is a bacterial infection that causes a variety of symptoms, including fever, muscle aches, and diarrhea. It is a common cause of death in people who have been exposed to the disease.

Tuberculosis is a bacterial infection that causes a variety of symptoms, including fever, muscle aches, and diarrhea. It is a common cause of death in people who have been exposed to the disease. What are the symptoms of


In [69]:
# Testing with finetuned model: prompt 2

prompt = "What causes dry eye?"
response = generate_response(my_model, my_tokenizer, prompt)
print("Generated response:", response)

Generated response: What causes dry eye? The causes of dry eye may include: Aging, ultraviolet (UV) rays, chemical fumes, or irritation of the eye's surface. Aging is the most common cause of dry eye. It causes the eye to dry up quickly, becoming very dry. Other causes of dry eye include: Colors that seem faded or faded. Darker-colored or undereye eyes may make it harder to see. The reason that people have dry eye may be because of these other factors.


In [70]:
# Testing with untuned model: prompt 2

prompt = "What causes dry eye?"
response = generate_response(model, tokenizer, prompt)
print("Generated response:", response)

Generated response: What causes dry eye?

The most common cause of dry eye is a lack of sunlight. Dry eye is caused by a lack of sunlight in the eye. The sun is not shining through the eye. The sun is not shining through the eye. The sun is not shining through the eye. The sun is not shining through the eye. The sun is not shining through the eye. The sun is not shining through the eye. The sun is not shining through the eye. The sun is not


In [71]:
# Testing with finetuned model: prompt 3

prompt = "What is the treatment for ligament injury?"
response = generate_response(my_model, my_tokenizer, prompt)
print("Generated response:", response)

Generated response: What is the treatment for ligament injury? Several types of surgery have been used to prevent tendon rupture. These techniques include hydrostatic and/or electric pulses, electrical nerve stimulation (ENRS), and electromyography (EMG). Hydrostatic and/or electric pulses stimulate the release of electrical impulses that recruit other nerves to the injured joint. Electromyography is a technique that uses small electronic devices to look inside the muscles of the ligament. A specially trained technician performs electromyography in the


In [72]:
# Testing with untuned model: prompt 3

prompt = "What is the treatment for ligament injury?"
response = generate_response(model, tokenizer, prompt)
print("Generated response:", response)

Generated response: What is the treatment for ligament injury?

The most common type of ligament injury is a torn ligament. The ligament is a small, irregular, or irregularly shaped piece of tissue that is attached to the bone. The ligament is usually broken or broken down. The ligament is usually repaired by a series of small, irregular, or irregularly shaped pieces of tissue.

The most common type of ligament injury is a torn ligament. The ligament is
